In [52]:
import argparse
import os,cv2,math
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as TF
from PIL import Image, ImageFilter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
train_info=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/train.csv")

if os.path.exists(f"/kaggle/working/train/")==False:
    os.mkdir(f"/kaggle/working/train/")
    os.mkdir(f"/kaggle/working/train/real/")
    os.mkdir(f"/kaggle/working/train/editada/")
    os.mkdir(f"/kaggle/working/test/")
    os.mkdir(f"/kaggle/working/test/real/")
    os.mkdir(f"/kaggle/working/test/editada/")
    os.mkdir(f"/kaggle/working/acc/")
    os.mkdir(f"/kaggle/working/error/")

for i in zip(train_info['image'],train_info['label'],range(len(train_info['image']))):
    
    image=cv2.resize(cv2.imread(f"/kaggle/input/cidaut-ai-fake-scene-classification-2024/Train/{i[0]}"),(256,256))
    
    if i[2]<=69:
        if i[1]=="real":
            cv2.imwrite(f"/kaggle/working/test/real/{i[0]}", cv2.cvtColor(image, cv2.COLOR_BGR2HSV))
        elif i[1]=="editada":
            cv2.imwrite(f"/kaggle/working/test/editada/{i[0]}", cv2.cvtColor(image, cv2.COLOR_BGR2HSV))
    
    if i[1]=="real":
        cv2.imwrite(f"/kaggle/working/train/real/{i[0]}", cv2.cvtColor(image, cv2.COLOR_BGR2HSV))
    elif i[1]=="editada":
        cv2.imwrite(f"/kaggle/working/train/editada/{i[0]}", cv2.cvtColor(image, cv2.COLOR_BGR2HSV))
        
        

In [50]:
workers = 2
batch_size = 10
nz = 100
num_epochs = 5
lr = 0.0001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

basic_mb_params = [
    # k, channels(c), repeats(t), stride(s), kernel_size(k)
    [1, 16, 1, 1, 3],
    [6, 24, 2, 2, 3],
    [6, 40, 2, 2, 5],
    [6, 80, 3, 2, 3],
    [6, 112, 3, 1, 5],
    [6, 192, 4, 2, 5],
    [6, 320, 1, 1, 3],
]

alpha, beta = 1.2, 1.1

scale_values = {
    # (phi, resolution, dropout)
    "b0": (0, 224, 0.2),
    "b1": (0.5, 240, 0.2),
    "b2": (1, 260, 0.3),
    "b3": (2, 300, 0.3),
    "b4": (3, 380, 0.4),
    "b5": (4, 456, 0.4),
    "b6": (5, 528, 0.5),
    "b7": (6, 600, 0.5),
}

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

trainset = torchvision.datasets.ImageFolder(root=f'/kaggle/working/train',transform=transform)
dataloader = torch.utils.data.DataLoader(trainset,batch_size=batch_size,shuffle=True,num_workers=2)

testset = torchvision.datasets.ImageFolder(root=f'/kaggle/working/test',transform=transform)
testloader = torch.utils.data.DataLoader(testset,batch_size=batch_size,shuffle=True,num_workers=2)

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [61]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, 
                 stride, padding, groups=1):
        super(ConvBlock, self).__init__()
        self.cnnblock = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size,
                                  stride, padding, groups=groups),
                        nn.BatchNorm2d(out_channels),
                        nn.SiLU())

    def forward(self, x):
        return self.cnnblock(x)
    
class MBBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
        stride, padding, ratio, reduction=2,
    ):
        super(MBBlock, self).__init__()
        # self.use_residual = in_channels == out_channels and stride == 1
        hidden_dim = in_channels * ratio
        self.expand = in_channels != hidden_dim

        # This is for squeeze and excitation block
        reduced_dim = int(in_channels / reduction)

        if self.expand:
            self.expand_conv = ConvBlock(in_channels, hidden_dim,
                kernel_size=3,stride=1,padding=1)

        self.conv = nn.Sequential(
                ConvBlock(hidden_dim,hidden_dim,kernel_size,
                  stride,padding,groups=hidden_dim),
                SqueezeExcitation(hidden_dim, reduced_dim),
                nn.Conv2d(hidden_dim, out_channels, 1),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, inputs):
        if self.expand:
          x = self.expand_conv(inputs)
        else:
          x = inputs
        return self.conv(x)
    
class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels, reduced_dim):
        super(SqueezeExcitation, self).__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # C x H x W -> C x 1 x 1
            nn.Conv2d(in_channels, reduced_dim, 1),
            nn.SiLU(),
            nn.Conv2d(reduced_dim, in_channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.se(x)

class EfficientNet(nn.Module):
    def __init__(self, model_name, output):
        super(EfficientNet, self).__init__()
        phi, resolution, dropout = scale_values[model_name]
        self.depth_factor, self.width_factor = alpha**phi, beta**phi
        self.last_channels = math.ceil(1280 * self.width_factor)
        self.avgpool= nn.AdaptiveAvgPool2d(1)
        self.feature_extractor()
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.last_channels, output),
        )

    def feature_extractor(self):
        channels = int(32 * self.width_factor)
        features = [ConvBlock(3, channels, 3, stride=2, padding=1)]
        in_channels = channels

        for k, c_o, repeat, s, n in basic_mb_params:
            # For numeric stability, we multiply and divide by 4
            out_channels = 4 * math.ceil(int(c_o * self.width_factor) / 4)
            num_layers = math.ceil(repeat * self.depth_factor)

            for layer in range(num_layers):
                if layer == 0:
                  stride = s
                else:
                  stride = 1
                features.append(
                        MBBlock(in_channels,out_channels,ratio=k,
                        stride=stride,kernel_size=n,padding=n// 2)
                    )
                in_channels = out_channels

        features.append(
            ConvBlock(in_channels, self.last_channels, 
            kernel_size=1, stride=1, padding=0)
        )
        self.extractor = nn.Sequential(*features)

    def forward(self, x):
        x = self.avgpool(self.extractor(x))
        FSC = self.classifier(self.flatten(x))
        return FSC,torch.argmax(FSC, dim=1)

In [71]:
model_name = 'b7'
output_class = 2 #for imagenet
EFF_NET = EfficientNet(model_name, output_class)
EFF_NET

EfficientNet(
  (avgpool): AdaptiveAvgPool2d(output_size=1)
  (extractor): Sequential(
    (0): ConvBlock(
      (cnnblock): Sequential(
        (0): Conv2d(3, 56, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(56, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
      )
    )
    (1): MBBlock(
      (conv): Sequential(
        (0): ConvBlock(
          (cnnblock): Sequential(
            (0): Conv2d(56, 56, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=56)
            (1): BatchNorm2d(56, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU()
          )
        )
        (1): SqueezeExcitation(
          (se): Sequential(
            (0): AdaptiveAvgPool2d(output_size=1)
            (1): Conv2d(56, 28, kernel_size=(1, 1), stride=(1, 1))
            (2): SiLU()
            (3): Conv2d(28, 56, kernel_size=(1, 1), stride=(1, 1))
            (4): Sigmoid()
          )
        )
  

In [76]:
criterion = nn.CrossEntropyLoss()

# CrossEntropyLoss
# MSELoss
# L1Loss
# BCELoss
# BCEWithLogitsLoss

optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr, betas=(beta1, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [ ]:
#netD.load_state_dict(torch.load("/kaggle/input/rafire_4/pytorch/default/1/rafire.pth"))
print(dataloader.dataset.classes)
z_w_=[]
high_rf,i_w,z_k=0,0,0
high_rf_=1

lab={
    'editada' : 0,
    'real' : 1
}
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')

while(i_w<4000):
    z_,z,z_w=0,0,{}
    correct=0
    for i, data in enumerate(dataloader, 0):
        real=data[0].to(device)
        label=data[1].to(device) #
        #print(data[0].shape)
        raw_label=torch.from_numpy(numpy.array([[0,1] if i==1 else [1,0] for i in data[1] ])).float().to(device)
        raw_output,output = EFF_NET(real)
        raw_output,output = raw_output.float() ,torch.tensor([lab[dataloader.dataset.classes[i]] for i in output.view(-1)])
        #print(raw_label,raw_output)
        err_real = criterion(raw_output,raw_label) #err_real.requires_grad = True
        optimizer.zero_grad()
        err_real.backward()
        optimizer.step()
        
    for i, data in enumerate(testloader, 0):
        real=data[0].to(device)
        label=data[1].to(device)
        raw_output,output = EFF_NET(real)
        output=torch.tensor([lab[dataloader.dataset.classes[i]] for i in output.view(-1)])
        z_+=len(output)
        correct+=(output==label).float().sum()
    
    z_add,z_total=0,0
    for i in fsc_submission.index:
        img=fsc_test[i].reshape((1, 3, 256, 256)).float().to(device)
        #print(FSC_net(img))
        raw_output,output = EFF_NET(img)
        if output.cpu().detach().numpy()==0:
            z_add+=1
        z_total+=1

    print(z_total,z_add)
        
    acc=100*correct/z_
    z_w_.append(acc)
    
    print(output,label)
    print(f"EPOCH: {z_k} LOSS_FSC: {err_real.item()} ACCURACY: {acc}")
    #print(z_)	
        
    if len(z_w_)>=3:
        if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_=[]
            print(optimizer.param_groups[0]["lr"])
            scheduler.step()
            print(optimizer.param_groups[0]["lr"])
            
    if acc>=high_rf:
        high_rf=acc
        torch.save(FSC_net.state_dict(),f'/kaggle/working/acc/{acc}.pth')

    if err_real.item()<=high_rf_: 
        high_rf_=err_real.item()
        torch.save(FSC_net.state_dict(),f'/kaggle/working/error/{err_real.item()}.pth')
    
    torch.save(FSC_net.state_dict(),f'/kaggle/working/{err_real.item()} {acc} {z_add}.pth')
    
    if z_k==100:
        break
    z_k+=1
    i_w+=1
    

['editada', 'real']
